# Transformer NMT with Pretrained Embeddings: FastText vs. BPEmb — NVIDIA RTX 3050  Ti GPU

 GPU (MPS) version
Same dataset and splits as the previous lab (ILCI subset of `ai4bharat/BPCC`, English → your language, 4 length bins).
Two Transformer models are trained, differing only in the embeddings used to initialise the source and target embedding layers:
1. **FastText** pretrained word vectors (https://fasttext.cc/docs/en/pretrained-vectors.html)
2. **BPEmb** subword embeddings (https://github.com/bheinzerling/bpemb)

Both are evaluated with sacreBLEU (overall and bin-wise), followed by the bin-wise analysis.


## 0. Environment

In [7]:
import sys, os, platform, shutil, importlib.util
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"   # unsupported ops run on the CPU instead of crashing (set before importing torch)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"       # avoids a common OpenMP crash on macOS

import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")                  # Apple GPU
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(sys.executable)
print("Machine:", platform.machine(), "| Python:", platform.python_version(), "| torch:", torch.__version__)
print("Training device:", device)
print(f"Free disk space here: {shutil.disk_usage('.').free / 1e9:.0f} GB  (FastText needs ~8 GB for the two .vec files)")

needed = ["sacrebleu", "bpemb", "gensim", "sentencepiece", "huggingface_hub", "pandas", "requests", "tqdm"]
missing = [p for p in needed if importlib.util.find_spec(p) is None]
if missing:
    raise ImportError("Missing packages. Run this in a terminal (same conda env), then restart the kernel:\n"
                      "    pip install " + " ".join(missing))
print("All packages available.")

/Users/adityakumar/Desktop/miniconda3/envs/DL/bin/python
Machine: arm64 | Python: 3.11.14 | torch: 2.9.1
Training device: mps
Free disk space here: 188 GB  (FastText needs ~8 GB for the two .vec files)
All packages available.


## 1. Configuration
Set `TARGET_LANG` to your native language (same as the previous lab). `ISO` maps it to the 2-letter code used by FastText/BPEmb.

In [8]:
TARGET_LANG = "hin_Deva"     # <-- change to your native language (same as the previous lab)
SOURCE_LANG = "eng_Latn"
TEST_FRAC   = 0.10
SEED        = 42
EMB_DIM     = 300            # FastText wiki vectors and BPEmb both offer 300-d vectors
BPEMB_VS    = 25000          # BPEmb subword vocabulary size
VOCAB_MAX   = 50000          # maximum vocabulary size per language

# ---- data caps (same as the previous notebooks, so the splits are identical) -----
MAX_TRAIN_PER_BIN = 5000     # cap on training sentences per bin (None = use all)
MAX_TEST_PER_BIN  = 500      # cap on test sentences per bin (None = use all)
VALID_PER_BIN     = 100      # validation sentences per bin (taken from the non-test data)

# ---- GPU training budget ---------------------------------------------------------
TRAIN_STEPS  = 6000          # per model. Try 300 first to see the speed, then raise it
BATCH_TOKENS = 6000          # tokens per batch (padded). Lower to 4000 if the Mac runs out of memory
SAVE_EVERY   = 1000          # checkpoint + validation interval
PEAK_LR      = 7e-4          # peak learning rate (linear warm-up, then inverse square-root decay)
WARMUP       = max(1, TRAIN_STEPS // 4)
MAX_LEN      = 128           # sentences longer than this many tokens are truncated
MAX_DECODE   = 200           # maximum generated length
DECODE_BATCH = 64            # sentences decoded at the same time

RUN_FASTTEXT = True
RUN_BPEMB    = True
WORKDIR     = "mt_emb_work"

ISO = {"hin_Deva": "hi", "guj_Gujr": "gu", "mar_Deva": "mr", "ben_Beng": "bn", "tam_Taml": "ta",
       "tel_Telu": "te", "kan_Knda": "kn", "mal_Mlym": "ml", "pan_Guru": "pa", "ory_Orya": "or",
       "asm_Beng": "as", "urd_Arab": "ur", "npi_Deva": "ne"}
TGT_ISO = ISO[TARGET_LANG]

all_results = {}
os.makedirs(WORKDIR, exist_ok=True)

## 2. Load the ILCI subset

In [9]:
from huggingface_hub import login, list_repo_files, hf_hub_download
import pandas as pd
login()

files = list_repo_files("ai4bharat/BPCC", repo_type="dataset")
cands = [f for f in files if "ilci" in f.lower() and TARGET_LANG in f]
print("Matching files:", cands)
assert cands, "No ILCI file found for this language; inspect `files` and adjust TARGET_LANG."

dfs = []
for f in cands:
    p = hf_hub_download("ai4bharat/BPCC", f, repo_type="dataset")
    if f.endswith(".parquet"):
        d = pd.read_parquet(p)
    elif f.endswith(".csv"):
        d = pd.read_csv(p)
    elif f.endswith(".tsv"):
        d = pd.read_csv(p, sep="\t")
    else:
        d = pd.read_json(p, lines=True)
    dfs.append(d)
df = pd.concat(dfs, ignore_index=True)
print(df.columns.tolist(), len(df))

if "src_lang" in df.columns:
    fwd = df[(df.src_lang == SOURCE_LANG) & (df.tgt_lang == TARGET_LANG)]
    rev = df[(df.src_lang == TARGET_LANG) & (df.tgt_lang == SOURCE_LANG)].rename(columns={"src": "tgt", "tgt": "src"})
    df = pd.concat([fwd, rev], ignore_index=True)
df = df[["src", "tgt"]].dropna()
df["src"] = df["src"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
df["tgt"] = df["tgt"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
df = df[(df.src != "") & (df.tgt != "")].drop_duplicates().reset_index(drop=True)
print("Sentence pairs:", len(df))

Matching files: ['ilci/hin_Deva.tsv']
['src_lang', 'tgt_lang', 'src', 'tgt'] 165649
Sentence pairs: 165649


## 3. Length-based bins and train / valid / test splits (same as the previous lab)
Length = number of words in the English sentence. bin1: ≤10, bin2: 11–20, bin3: 21–29, bin4: ≥30 (length 20 is placed in bin2 so every sentence has a bin). Each bin is split separately, with the same seed and caps as the previous notebook, so the splits are identical. A small validation set (needed by OpenNMT-py) is taken from the non-test data.

In [10]:
def get_bin(n):
    if n <= 10: return "bin1"
    if n <= 20: return "bin2"
    if n < 30:  return "bin3"
    return "bin4"

df["len"] = df["src"].str.split().str.len()
df["bin"] = df["len"].apply(get_bin)

train_parts, valid_parts, test_parts = [], [], []
for b, g in df.groupby("bin"):
    g = g.sample(frac=1.0, random_state=SEED)
    n_test = max(1, int(round(len(g) * TEST_FRAC)))
    te, rest = g.iloc[:n_test], g.iloc[n_test:]
    va, tr = rest.iloc[:VALID_PER_BIN], rest.iloc[VALID_PER_BIN:]
    if MAX_TEST_PER_BIN is not None:  te = te.iloc[:MAX_TEST_PER_BIN]
    if MAX_TRAIN_PER_BIN is not None: tr = tr.iloc[:MAX_TRAIN_PER_BIN]
    test_parts.append(te); valid_parts.append(va); train_parts.append(tr)

train_df = pd.concat(train_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
valid_df = pd.concat(valid_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
test_df  = pd.concat(test_parts).reset_index(drop=True)

print(pd.DataFrame({"train": train_df["bin"].value_counts().sort_index(),
                    "valid": valid_df["bin"].value_counts().sort_index(),
                    "test":  test_df["bin"].value_counts().sort_index()}))

      train  valid  test
bin                     
bin1   5000    100   500
bin2   5000    100   500
bin3   5000    100   500
bin4   5000    100   500


## 4. Helper functions (tokenisation, downloads, vocabularies, embedding matrices, Transformer, GPU training/decoding, BLEU)

In [14]:
import re, pickle, math, random, time, numpy as np, requests
from collections import Counter
from tqdm.auto import tqdm
from sacrebleu.metrics import BLEU
import torch.nn as nn
import torch.nn.functional as F

PUNCT = re.compile(r"([.,!?;:()\[\]{}\"%/।॥])")
def tok(s):
    """Simple word tokeniser: separates punctuation (incl. the Devanagari danda) and splits on whitespace."""
    return PUNCT.sub(r" \1 ", s).split()

def download(url, dest, chunk=1 << 23):
    """Resumable download (no wget needed on macOS). Skips the file if it already exists."""
    if os.path.exists(dest):
        print("Already downloaded:", dest); return
    tmp = dest + ".part"
    have = os.path.getsize(tmp) if os.path.exists(tmp) else 0
    headers = {"Range": f"bytes={have}-"} if have else {}
    with requests.get(url, stream=True, headers=headers, timeout=60) as r:
        if r.status_code == 416:                      # .part is already complete
            os.replace(tmp, dest); return
        r.raise_for_status()
        resumed = have > 0 and r.status_code == 206
        total = int(r.headers.get("Content-Length", 0)) + (have if resumed else 0)
        with open(tmp, "ab" if resumed else "wb") as f, tqdm(total=total or None, initial=have if resumed else 0,
                                                              unit="B", unit_scale=True, desc=os.path.basename(dest)) as bar:
            for part in r.iter_content(chunk):
                f.write(part); bar.update(len(part))
    os.replace(tmp, dest)

def load_fasttext_subset(path, wanted, cache_path):
    """Stream a (huge) FastText .vec file and keep only the vectors of `wanted` words. The result is cached."""
    wanted = set(wanted) | {w.lower() for w in wanted}
    if os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            c = pickle.load(f)
        if wanted <= c["wanted"]:
            print("Using cached vectors:", cache_path); return c["vecs"]
    vecs = {}
    with open(path, "rb") as f:
        f.readline()                                       # header line "<n_words> <dim>"
        for line in tqdm(f, desc=os.path.basename(path), unit=" words"):
            w, _, rest = line.partition(b" ")
            w = w.decode("utf-8", "ignore")
            if w in wanted:
                v = np.array(rest.split(), dtype="float32")
                if v.shape[0] == EMB_DIM:
                    vecs[w] = v
    with open(cache_path, "wb") as f:
        pickle.dump({"wanted": wanted, "vecs": vecs}, f)
    return vecs

# ---- vocabulary and embedding matrices --------------------------------------------
PAD, UNK, BOS, EOS = 0, 1, 2, 3
SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

def build_vocab(token_lists):
    counts = Counter(t for toks in token_lists for t in toks)
    itos = SPECIALS + [w for w, _ in counts.most_common(VOCAB_MAX - len(SPECIALS)) if w not in SPECIALS]
    return itos, {w: i for i, w in enumerate(itos)}

def build_embedding_matrix(itos, vec_dict, label):
    """Row i = pretrained vector of the i-th vocabulary entry (random init if not available)."""
    rng = np.random.RandomState(SEED)
    mat = rng.normal(0, 0.1, size=(len(itos), EMB_DIM)).astype("float32")
    mat[PAD] = 0.0
    found = 0
    for i, w in enumerate(itos[len(SPECIALS):], start=len(SPECIALS)):
        v = vec_dict.get(w)
        if v is None:
            v = vec_dict.get(w.lower())
        if v is not None:
            mat[i] = v; found += 1
    print(f"{label}: {found}/{len(itos) - len(SPECIALS)} vocabulary entries found in the pretrained embeddings")
    return torch.from_numpy(mat)

# ---- data -> tensors --------------------------------------------------------------
def to_ids(tokens, stoi):
    return [stoi.get(t, UNK) for t in tokens[:MAX_LEN - 1]]

def make_pairs(src_tok, tgt_tok, src_stoi, tgt_stoi):
    """(source ids + </s>, target ids)"""
    return [(to_ids(s, src_stoi) + [EOS], to_ids(t, tgt_stoi)) for s, t in zip(src_tok, tgt_tok)]

def pad_batch(seqs):
    out = torch.zeros(len(seqs), max(len(s) for s in seqs), dtype=torch.long)
    for r, s in enumerate(seqs):
        out[r, :len(s)] = torch.tensor(s, dtype=torch.long)
    return out

def make_batches(pairs, max_tokens):
    """Group sentences of similar length; every batch holds at most max_tokens (padded) tokens."""
    length = lambda i: max(len(pairs[i][0]), len(pairs[i][1]) + 1)
    order = sorted(range(len(pairs)), key=length)
    batches, cur, cur_max = [], [], 0
    for i in order:
        L = length(i)
        if cur and max(cur_max, L) * (len(cur) + 1) > max_tokens:
            batches.append(cur); cur, cur_max = [], 0
        cur.append(i); cur_max = max(cur_max, L)
    if cur:
        batches.append(cur)
    return batches

def collate(pairs, idx):
    src = pad_batch([pairs[i][0] for i in idx])
    tgt_in = pad_batch([[BOS] + pairs[i][1] for i in idx])
    tgt_out = pad_batch([pairs[i][1] + [EOS] for i in idx])
    return src, tgt_in, tgt_out

# ---- the Transformer --------------------------------------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class NMT(nn.Module):
    def __init__(self, src_emb, tgt_emb, heads=6, ff=1200, layers=3, dropout=0.1):
        super().__init__()
        d = src_emb.size(1)
        self.scale = math.sqrt(d)
        self.src_emb = nn.Embedding.from_pretrained(src_emb.clone(), freeze=False, padding_idx=PAD)
        self.tgt_emb = nn.Embedding.from_pretrained(tgt_emb.clone(), freeze=False, padding_idx=PAD)
        self.pos = PositionalEncoding(d)
        self.drop = nn.Dropout(dropout)
        enc_layer = nn.TransformerEncoderLayer(d, heads, ff, dropout, batch_first=True, norm_first=True)
        dec_layer = nn.TransformerDecoderLayer(d, heads, ff, dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, layers, norm=nn.LayerNorm(d), enable_nested_tensor=False)
        self.decoder = nn.TransformerDecoder(dec_layer, layers, norm=nn.LayerNorm(d))
        self.out = nn.Linear(d, tgt_emb.size(0))
        for module in (self.encoder, self.decoder, self.out):      # Glorot init for the layers (embeddings keep the pretrained values)
            for p in module.parameters():
                if p.dim() > 1:
                    nn.init.xavier_uniform_(p)

    def encode(self, src):
        pad_mask = src == PAD
        x = self.drop(self.pos(self.src_emb(src) * self.scale))
        return self.encoder(x, src_key_padding_mask=pad_mask), pad_mask

    def decode(self, tgt_in, memory, src_pad_mask):
        T = tgt_in.size(1)
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tgt_in.device), diagonal=1)
        x = self.drop(self.pos(self.tgt_emb(tgt_in) * self.scale))
        h = self.decoder(x, memory, tgt_mask=causal, tgt_key_padding_mask=(tgt_in == PAD),
                         memory_key_padding_mask=src_pad_mask)
        return self.out(h)

    def forward(self, src, tgt_in):
        memory, pad_mask = self.encode(src)
        return self.decode(tgt_in, memory, pad_mask)

# ---- training ---------------------------------------------------------------------
def lr_at(step):
    return PEAK_LR * min(step / WARMUP, math.sqrt(WARMUP / step))

@torch.no_grad()
def validation_loss(model, pairs):
    model.eval()
    total = torch.zeros((), device=device)
    count = torch.zeros((), device=device)
    for idx in make_batches(pairs, BATCH_TOKENS):
        src, tgt_in, tgt_out = [t.to(device) for t in collate(pairs, idx)]
        logits = model(src, tgt_in)
        total += F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1),
                                 ignore_index=PAD, reduction="sum")
        count += (tgt_out != PAD).sum()
    return (total / count).item()

def train_model(exp, model, train_pairs, valid_pairs):
    ckpt_path = f"{exp}/ckpt.pt"
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=PEAK_LR, betas=(0.9, 0.98), eps=1e-9)
    step = 0
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); step = ck["step"]
        print("Resuming from step", step)
    if step >= TRAIN_STEPS:
        print(f"Finished checkpoint found ({ckpt_path}); skipping training. Delete it to retrain.")
        return

    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)
    batches = make_batches(train_pairs, BATCH_TOKENS)
    print(f"{len(train_pairs)} training pairs in {len(batches)} batches per epoch")
    bar = tqdm(total=TRAIN_STEPS, initial=step, desc="training")
    run_loss, run_n = torch.zeros((), device=device), 0       # loss accumulated on the GPU, read every 100 steps
    model.train()
    while step < TRAIN_STEPS:
        random.shuffle(batches)
        for idx in batches:
            step += 1
            for g in opt.param_groups:
                g["lr"] = lr_at(step)
            src, tgt_in, tgt_out = [t.to(device) for t in collate(train_pairs, idx)]
            logits = model(src, tgt_in)
            loss = loss_fn(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            run_loss += loss.detach(); run_n += 1
            bar.update(1)
            if step % 100 == 0:
                bar.set_postfix(loss=f"{run_loss.item() / run_n:.3f}", lr=f"{lr_at(step):.1e}")
                run_loss.zero_(); run_n = 0
            if step % SAVE_EVERY == 0 or step == TRAIN_STEPS:
                val = validation_loss(model, valid_pairs)
                print(f"\nstep {step}: validation loss {val:.3f}")
                torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "step": step}, ckpt_path)
                model.train()
            if step >= TRAIN_STEPS:
                break
    bar.close()

# ---- decoding (batched, greedy) ---------------------------------------------------
@torch.no_grad()
def greedy_decode(model, src_seqs, itos, batch_size=DECODE_BATCH):
    model.eval()
    order = sorted(range(len(src_seqs)), key=lambda i: len(src_seqs[i]))      # similar lengths together
    results = [None] * len(src_seqs)
    for b in tqdm(range(0, len(order), batch_size), desc="decoding"):
        idx = order[b:b + batch_size]
        src = pad_batch([src_seqs[i] for i in idx]).to(device)
        memory, pad_mask = model.encode(src)
        max_out = min(int(src.size(1) * 1.5) + 10, MAX_DECODE)
        ys = torch.full((len(idx), 1), BOS, dtype=torch.long, device=device)
        done = torch.zeros(len(idx), dtype=torch.bool, device=device)
        for _ in range(max_out):
            logits = model.decode(ys, memory, pad_mask)[:, -1, :]
            logits[:, PAD] = float("-inf")
            logits[:, BOS] = float("-inf")
            nxt = logits.argmax(dim=-1)
            nxt = torch.where(done, torch.full_like(nxt, PAD), nxt)
            ys = torch.cat([ys, nxt.unsqueeze(1)], dim=1)
            done = done | (nxt == EOS)
            if bool(done.all()):
                break
        for row, i in zip(ys[:, 1:].tolist(), idx):
            toks = []
            for t in row:
                if t in (EOS, PAD):
                    break
                toks.append(itos[t])
            results[i] = toks
    return results

def run_nmt(exp, train_tok, valid_tok, test_src_tok, src_vecs_fn, tgt_vecs_fn):
    """train_tok / valid_tok = (source token lists, target token lists). Returns the hypotheses as token lists."""
    os.makedirs(exp, exist_ok=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    src_itos, src_stoi = build_vocab(train_tok[0])
    tgt_itos, tgt_stoi = build_vocab(train_tok[1])
    print(f"Vocabulary sizes: source {len(src_itos)}, target {len(tgt_itos)}")
    src_mat = build_embedding_matrix(src_itos, src_vecs_fn(src_itos), "source")
    tgt_mat = build_embedding_matrix(tgt_itos, tgt_vecs_fn(tgt_itos), "target")

    train_pairs = make_pairs(*train_tok, src_stoi, tgt_stoi)
    valid_pairs = make_pairs(*valid_tok, src_stoi, tgt_stoi)
    model = NMT(src_mat, tgt_mat)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M | device: {device}")

    t0 = time.time()
    train_model(exp, model, train_pairs, valid_pairs)
    print(f"Training time: {(time.time() - t0) / 60:.1f} min")

    test_src = [to_ids(s, src_stoi) + [EOS] for s in test_src_tok]
    t0 = time.time()
    hyps = greedy_decode(model, test_src, tgt_itos)
    print(f"Decoding time: {(time.time() - t0) / 60:.1f} min")
    del model
    if device.type == "mps":
        torch.mps.empty_cache()
    return hyps

# ---- evaluation -------------------------------------------------------------------
bleu = BLEU(tokenize="none")   # texts are word-tokenised with tok() on both hypothesis and reference sides

def evaluate(hyps_text, name):
    """hyps_text: detokenised hypotheses (one per test sentence). Returns bin-wise + overall BLEU."""
    hyp = [" ".join(tok(h)) for h in hyps_text]
    ref = [" ".join(tok(r)) for r in test_df["tgt"]]
    res = {"overall": bleu.corpus_score(hyp, [ref]).score}
    for b in ["bin1", "bin2", "bin3", "bin4"]:
        idx = (test_df["bin"] == b).values
        res[b] = bleu.corpus_score([h for h, k in zip(hyp, idx) if k], [[r for r, k in zip(ref, idx) if k]]).score
    print(name, {k: round(v, 2) for k, v in res.items()})
    return res

## 5. Experiment 1: FastText embeddings
Text is word-tokenised; the Transformer's embedding layers (source: English, target: your language) are initialised from the pretrained FastText `wiki.<lang>.vec` vectors. Words missing from FastText get random vectors (a lower-cased lookup is tried first). Both embedding tables are fine-tuned during training.

In [15]:
if RUN_FASTTEXT:
    exp = f"{WORKDIR}/fasttext_gpu"
    os.makedirs(exp, exist_ok=True)

    tokenise = lambda texts: [tok(s) for s in texts]
    train_tok = (tokenise(train_df["src"]), tokenise(train_df["tgt"]))
    valid_tok = (tokenise(valid_df["src"]), tokenise(valid_df["tgt"]))
    test_src_tok = tokenise(test_df["src"])

    # pretrained FastText vectors (English and target language); resumable, skipped if present
    for code_ in ["en", TGT_ISO]:
        download(f"https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.{code_}.vec", f"{WORKDIR}/wiki.{code_}.vec")

    ft_vecs = lambda code_: (lambda itos: load_fasttext_subset(f"{WORKDIR}/wiki.{code_}.vec", set(itos),
                                                               f"{exp}/ft_subset_{code_}.pkl"))
    ft_tokens = run_nmt(exp, train_tok, valid_tok, test_src_tok, ft_vecs("en"), ft_vecs(TGT_ISO))
    ft_hyps = [" ".join(t) for t in ft_tokens]
    all_results["FastText"] = evaluate(ft_hyps, "FastText")

Already downloaded: mt_emb_work/wiki.en.vec
Already downloaded: mt_emb_work/wiki.hi.vec
Vocabulary sizes: source 34658, target 38591
Using cached vectors: mt_emb_work/fasttext_gpu/ft_subset_en.pkl
source: 29108/34654 vocabulary entries found in the pretrained embeddings
Using cached vectors: mt_emb_work/fasttext_gpu/ft_subset_hi.pkl
target: 23581/38587 vocabulary entries found in the pretrained embeddings
Model parameters: 41.2M | device: mps
20000 training pairs in 92 batches per epoch


training:   0%|          | 0/6000 [00:00<?, ?it/s]

RuntimeError: MPS backend out of memory (MPS allocated: 7.37 GiB, other allocations: 22.68 GiB, max allowed: 30.19 GiB). Tried to allocate 883.28 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

## 6. Experiment 2: BPEmb subword embeddings
The text is first encoded into BPEmb subwords (with the BPEmb sentencepiece models for English and the target language). The embedding layers are initialised with the BPEmb vectors of those subwords. Hypotheses are decoded back to text with BPEmb before scoring. (BPEmb lower-cases text and maps every digit to `0` when encoding; this is part of BPEmb and can cost it a little BLEU on sentences with numbers.)

In [16]:
if RUN_BPEMB:
    from bpemb import BPEmb

    bp_src = BPEmb(lang="en", vs=BPEMB_VS, dim=EMB_DIM)       # downloads the model on first use
    bp_tgt = BPEmb(lang=TGT_ISO, vs=BPEMB_VS, dim=EMB_DIM)

    def bpemb_dict(b):
        kv = b.emb
        try:
            idx = kv.key_to_index
        except AttributeError:
            idx = {w: v.index for w, v in kv.vocab.items()}
        return {w: kv.vectors[i] for w, i in idx.items()}

    exp = f"{WORKDIR}/bpemb_gpu"
    os.makedirs(exp, exist_ok=True)

    encode_src = lambda texts: [bp_src.encode(s) for s in texts]
    encode_tgt = lambda texts: [bp_tgt.encode(s) for s in texts]
    train_tok = (encode_src(train_df["src"]), encode_tgt(train_df["tgt"]))
    valid_tok = (encode_src(valid_df["src"]), encode_tgt(valid_df["tgt"]))
    test_src_tok = encode_src(test_df["src"])

    src_dict, tgt_dict = bpemb_dict(bp_src), bpemb_dict(bp_tgt)
    bp_tokens = run_nmt(exp, train_tok, valid_tok, test_src_tok, lambda itos: src_dict, lambda itos: tgt_dict)
    bp_hyps = [bp_tgt.decode(t) for t in bp_tokens]
    all_results["BPEmb"] = evaluate(bp_hyps, "BPEmb")

downloading https://nlp.h-its.org/bpemb/en/en.wiki.bpe.vs25000.model








100%|██████████| 661443/661443 [00:00<00:00, 954156.17B/s]


downloading https://nlp.h-its.org/bpemb/en/en.wiki.bpe.vs25000.d300.w2v.bin.tar.gz





























































































































































































































100%|██████████| 28009055/28009055 [00:33<00:00, 832226.63B/s]


downloading https://nlp.h-its.org/bpemb/hi/hi.wiki.bpe.vs25000.model


...

downloading https://nlp.h-its.org/bpemb/hi/hi.wiki.bpe.vs25000.d300.w2v.bin.tar.gz










































































































































































































































































































































































































































































































































100%|██████████| 28060238/28060238 [01:17<00:00, 360661.15B/s] 


Vocabulary sizes: source 16559, target 17879
source: 16520/16555 vocabulary entries found in the pretrained embeddings
target: 17822/17875 vocabulary entries found in the pretrained embeddings
Model parameters: 23.3M | device: mps
20000 training pairs in 107 batches per epoch


training:   0%|          | 0/6000 [00:00<?, ?it/s]

RuntimeError: MPS backend out of memory (MPS allocated: 6.96 GiB, other allocations: 23.13 GiB, max allowed: 30.19 GiB). Tried to allocate 407.85 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

## 7. Bin-wise BLEU comparison

In [ ]:
counts = test_df["bin"].value_counts().sort_index()
lens = test_df.groupby("bin")["len"].mean().round(1)
results = pd.DataFrame(all_results).T.round(2)
results = results[["bin1", "bin2", "bin3", "bin4", "overall"]]
print("Test sentences per bin:", counts.to_dict())
print("Average source length per bin:", lens.to_dict())
results

## 8. Bin-wise result analysis

In [ ]:
for name in results.index:
    r = results.loc[name, ["bin1", "bin2", "bin3", "bin4"]]
    d = r.diff().dropna()
    trend = ("decreases steadily with length" if (d < 0).all() else
             "increases steadily with length" if (d > 0).all() else "is not monotonic in length")
    print(f"{name}: best bin = {r.idxmax()} ({r.max():.2f}), worst bin = {r.idxmin()} ({r.min():.2f}); BLEU {trend}.")

if len(results) == 2:
    print()
    for b in ["bin1", "bin2", "bin3", "bin4", "overall"]:
        w = results[b].idxmax()
        print(f"{b}: {w} is better (FastText {results.loc['FastText', b]:.2f} vs BPEmb {results.loc['BPEmb', b]:.2f})")

**Analysis (fill in after running):** Use the table and printed summary above to discuss (a) how BLEU changes with sentence length for each embedding type, (b) which embedding is better overall and in which bins, (c) why: FastText works on whole words, so rare/unseen words become `<unk>` and many vocabulary entries have no pretrained vector, while BPEmb's subwords cover any word and give a small, fully-covered vocabulary, which usually helps on long sentences and morphologically rich languages, (d) the number of test sentences per bin (small bins give noisy scores). Note that this run uses a small model, a capped training set, a limited number of training steps and greedy decoding, so absolute BLEU will be low; compare the two embeddings with each other rather than with published numbers.